In [1]:
!pip install pyarrow==2 awswrangler

     |████████████████████████████████| 17.7 MB 19.7 MB/s eta 0:00:01
     |████████████████████████████████| 211 kB 118.9 MB/s eta 0:00:01
     |████████████████████████████████| 207 kB 118.8 MB/s eta 0:00:01
     |████████████████████████████████| 94 kB 624 kB/s s eta 0:00:01
     |████████████████████████████████| 43 kB 4.2 MB/s  eta 0:00:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 5.0.0
    Uninstalling pyarrow-5.0.0:
      Successfully uninstalled pyarrow-5.0.0
You should consider upgrading via the '/home/ec2-user/anaconda3/envs/python3/bin/python -m pip install --upgrade pip' command.


In [2]:
from datetime import datetime, timedelta, date
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
import random
import awswrangler as wr

In [3]:
sql = f"""
select distinct pd_sold.year_week_sold
        ,pd_sold.year_month_id
        ,pd_sold.id_shop
        ,pd_sold.id_warehouse
        
        ---- total no. of style for new arivals [1 weeks] ----    
        ,case when pd_launched.product_launched is null then 0 
            else pd_launched.product_launched end as tot_product_launched 
        ---- total no. of style for exiting products  ----
        ,case when sum(exist_styles.active_product) is null then 0
            else sum(exist_styles.active_product) end as tot_product_existing
            

            
 --total no of style
--        , sum((exist_styles.active_product)+ pd_launched.product_launched
--        as total_products        
        
 ---- sales unit new products  ----
        ,case when sum(pd_sold.net_units_sold) = 0 then 0
        when sum(pd_sold.new_pd_sold) <= 0 then 0
            else sum(pd_sold.new_pd_sold) 
            end as new_pd_sold
         
         ---- sales unit new products  ----
        ,case when sum(pd_sold.net_units_sold) = 0 then 0
        when sum(pd_sold.old_pd_sold) <= 0 then 0
            else sum(pd_sold.old_pd_sold) 
            end as old_pd_sold
            
        ---- ** PROPORTION NEW PRODUCT **  ----
        ,case when sum(pd_sold.net_units_sold) = 0 then 0
        when sum(pd_sold.new_pd_sold) <= 0 then 0
            else sum(pd_sold.new_pd_sold)/sum(pd_sold.net_units_sold)
            end as prop_new_pd
        ---- ** PROPORTION OLD PRODUCT **  ----
        ,case when sum(pd_sold.net_units_sold) = 0 then 0
        when sum(pd_sold.old_pd_sold) <= 0 then 0
            else sum(pd_sold.old_pd_sold)/sum(pd_sold.net_units_sold)
            end as prop_existing_pd


from
----/* PRODUCT SOLD FOR EXISTING STYLES */----
( select distinct t.year_week_sold
        ,t.year_month_id
        ,t.id_shop
        ,t.id_warehouse
        ,t.id_store
        --- new & old - normal & lookbook ---
        ,case when product_age = 'new' then sum(t.tot_net_units_sold) else 0 end as new_pd_sold
        ,case when product_age = 'old' then sum(t.tot_net_units_sold) else 0 end as old_pd_sold
        ,sum(t.tot_net_units_sold) as net_units_sold
        from 
        (--- fact sales ---
        select dp2.id_product 
        ,fso2.id_shop
        --- case erply_store_id ---
        ,case when fso2.id_store = '5b989110a9afac06f24b80ce' then '231'
            when fso2.id_store = '5b55a681868e636a8ab1eaa2' then '261'
            when fso2.id_store = '5bb2e6ac1e5abf2252df039c' then '251'
            when fso2.id_store = '5bbedd0bc3cef17af5b1a3ab' then '11'
            when fso2.id_store = '5b9890f4d49bab0743f5689e' then '241'
            when fso2.id_store = '5a7c011b6878b592402d76bd' then '371'
            else fso2.id_store end as id_store
        ,case when id_shop = '2' and year_week_id < '202017' then '1'
            when id_shop = '2' and year_week_id >= '202017' then '11'
            when id_shop = '11' and year_week_id < '202038' then '1'
            when id_shop = '11' and year_week_id >= '202038' then '11'
            when id_shop = '10' then '1'
            when id_shop = '12' then '1'
            when id_shop = '4' then '1'
            when id_shop = '14' then '11'
            else id_shop end as id_warehouse
        ,ca.year_week_id as year_week_sold
        ,ca.year_month_id
        --- first date available ---
        ,min(pd_avl.year_week_available) as year_week_avl
        --- product age ---
        ,case when min(pd_avl.year_week_available) = ca.year_week_id then 'new'
        --when ca.year_week_id - year_week_avl = 1 then 'new' 
        --when ca.year_week_id - year_week_avl = 2 then 'new' 
        else 'old' end as product_age
        ,coalesce (sum(CASE WHEN fso2.transaction_type ='Return' THEN -fso2.product_quantity ELSE fso2.product_quantity END),0) as tot_net_units_sold
        from dwh.fact_sales_offline fso2
        ---/* LOOKBOOK COLLECTION */---
        left join (select distinct id_product
        ,sku_complete
        ,date_released
        from dwh.dim_product
        where date_released between DATE_TRUNC('month', NOW() - INTERVAL '6' month) AND current_date
        and id_product is not null
        and original_price_th is not NULL 
        and date_released is not NULL 
        and parent_product_line is not null
        and product_name is not null
        and product_line is not null
        and original_price_th != 0
        and henry_category_1 not in ('Accessories', 'Bags', 'Cosmetics', 'Miscellaneous')
        and parent_product_line not in ('Cosmetics','Accessories','Free Gift','3rd Party')
        and brand in ('Alita','Basics','Pomelo', 'BEET by Pomelo', 'Holiday Collection', 'Pomelo x Tex Saverio')
        and id_product not in (19, 41, 14196, 14197, 14198, 14199, 14200, 14201, 14202, 14203, 14204, 14205, 14206, 14207, 14208, 17131, 17132, 17654,17182)
        and active = 1
        ) dp2
        on fso2.id_product = dp2.id_product
        ----/* Available date from NS */----
        left join (select sku_complete
        ,to_location_name
        ,case when to_id = 14 then 11
            when to_id = 29 then 211
            when to_id = 31 then 221
            when to_id = 69 then 231
            when to_id = 70 then 241
            when to_id = 71 then 251
            when to_id = 72 then 261
            when to_id = 73 then 281
            when to_id = 81 then 12
            when to_id = 92 then 311
            when to_id = 93 then 341
            when to_id = 95 then 321
            when to_id = 96 then 291
            when to_id = 99 then 301
            when to_id = 102 then 361
            when to_id = 103 then 331
            when to_id = 115 then 351
            when to_id = 116 then 381
            when to_id = 130 then 15
            when to_id = 131 then 371
            when to_id = 132 then 32
            when to_id = 159 then 391
            when to_id = 163 then 35
            when to_id = 165 then 42
            when to_id = 168 then 401
            when to_id = 175 then 411
            when to_id = 177 then 431
            when to_id = 178 then 111
            when to_id = 182 then 391
            when to_id = 186 then 441
            else null
            end as id_store
        ,min(ca.year_week_id) as year_week_available
        ,min(date(date_received)) as first_available_date
        FROM dwh.fact_ns_transfer_order ns
        ----/* YEAR WEEK ID FOR FIRST AVAILABLE DATE  */----
        left join (select full_date,
		case when ca.week_of_year_number  in (1,2,3,4,5,6,7,8,9) 
        then concat(cast(ca.year_id as varchar),'0',CAST(ca.week_of_year_number AS VARCHAR))
             when ca.week_of_year_number = 53 or ca.week_of_year_number = 54
              then concat(cast(ca.year_id+1 as varchar),'01')
            else concat(cast(ca.year_id as varchar),cast(ca.week_of_year_number as varchar)) 
            end as year_week_id
        from dwh.dim_calendar ca) ca
        on  date(ns.date_received) = ca.full_date 
        where 3 is not null
        GROUP BY 1,2,3
        order by sku_complete
        ) pd_avl
        on cast(dp2.sku_complete as varchar) = pd_avl.sku_complete and cast(pd_avl.id_store as varchar) = fso2.id_store 
        ----/* YEAR WEEK ID FOR TRANSACTION  */----
        left join (select full_date
        ,case when ca.month_id in (1,2,3,4,5,6,7,8,9) 
            then concat(cast(ca.year_id as varchar),'0',CAST(ca.month_id AS VARCHAR))
            else concat(cast(ca.year_id as varchar),cast(ca.week_of_year_number as varchar)) end as year_month_id
        , 
			case when ca.week_of_year_number  in (1,2,3,4,5,6,7,8,9) 
        then concat(cast(ca.year_id as varchar),'0',CAST(ca.week_of_year_number AS VARCHAR))
             when ca.week_of_year_number = 53 or ca.week_of_year_number = 54
              then concat(cast(ca.year_id+1 as varchar),'01')
            else concat(cast(ca.year_id as varchar),cast(ca.week_of_year_number as varchar)) 
            end as year_week_id
        from dwh.dim_calendar ca) ca
        on  date(fso2.transaction_time) = ca.full_date
        where fso2.revenue_usd >0
        ---- NOTE: Marketing spend started 2019 only ----
        --and transaction_time >= date('2019-01-01')
        and sales_id is not null
        and dp2.id_product is not null
        and pd_avl.year_week_available is not null
        group by dp2.id_product
            ,fso2.id_shop
            ,fso2.id_store
            ,ca.year_week_id
            ,ca.year_month_id
 		having ca.year_week_id >= min(pd_avl.year_week_available)) t
        group by t.year_week_sold
            ,t.id_shop
            ,t.year_month_id
            ,t.id_warehouse
            ,t.id_store
            ,product_age) pd_sold
        left join 
        (select ca.year_week_id
        ,case when fisom.id_shop is null then '1' else fisom.id_shop end as id_shop
        ,case when id_store = '5b989110a9afac06f24b80ce' then '231'
            when id_store = '5b55a681868e636a8ab1eaa2' then '261'
            when id_store = '5bb2e6ac1e5abf2252df039c' then '251'
            when id_store = '5bbedd0bc3cef17af5b1a3ab' then '11'
            when id_store = '5b9890f4d49bab0743f5689e' then '241'
            when id_store = '5a7c011b6878b592402d76bd' then '371'
            else id_store
            end as id_store
        ,count(distinct dp2.id_product) as active_product
        from dwh.fact_inventory_snapshot_offline_master fisom 
        left join (select distinct id_product
        from dwh.dim_product dp
        where dp.date_released between  DATE_TRUNC('month', NOW() - INTERVAL '6' month) AND current_date
        and dp.id_product is not null
        and dp.original_price_th is not NULL 
        and dp.date_released is not NULL 
        and dp.parent_product_line is not null
        and dp.product_name is not null
        and dp.product_line is not null
        and dp.original_price_th != 0
        and dp.henry_category_1 not in ('Accessories', 'Bags', 'Cosmetics', 'Miscellaneous')
        --,'Lingerie Bodysuit', 'Sportswear Bottoms', 'Sportswear Outerwear', 'Sportswear Tops', 'Swimwear')
        and dp.parent_product_line not in ('Cosmetics','Accessories','Free Gift','3rd Party')
        and brand in ('Alita','Basics','Pomelo', 'BEET by Pomelo', 'Holiday Collection', 'Pomelo x Tex Saverio')
        and dp.id_product not in (19, 41, 14196, 14197, 14198, 14199, 14200, 14201, 14202, 14203, 14204, 14205, 14206, 14207, 14208, 17131, 17132, 17654,17182)
        and dp.active = 1) dp2
        on fisom.id_product = dp2.id_product	
        left join (select full_date
        , case when ca.week_of_year_number  in (1,2,3,4,5,6,7,8,9) 
        then concat(cast(ca.year_id as varchar),'0',CAST(ca.week_of_year_number AS VARCHAR))
             when ca.week_of_year_number = 53 or ca.week_of_year_number = 54
              then concat(cast(ca.year_id+1 as varchar),'01')
            else concat(cast(ca.year_id as varchar),cast(ca.week_of_year_number as varchar)) 
            end as year_week_id
        from dwh.dim_calendar ca) ca
        on fisom.snapshot_date = ca.full_date
        where active = 1
        and fisom.snapshot_date BETWEEN DATE_TRUNC('month', NOW() - INTERVAL '6' month) AND current_date
        and id_store not in ('1','201', '25', '271')	
        group by id_shop
            ,id_store
            ,ca.year_week_id) exist_styles
        on pd_sold.year_week_sold = exist_styles.year_week_id and pd_sold.id_store = exist_styles.id_store     
       
                ----/* NEW PRODUCT LAUNCHED */----
        ---- get total product launched in that week ----
        left join (select distinct year_week_available
        ,id_store as id_store
        ,sum(tot_product_launched) as product_launched 
        from (select distinct avl_store.year_week_available
        ,avl_store.id_store as id_store
        ,count(distinct avl_store.id_product) as tot_product_launched
        from (select dp2.id_product
        ,dp2.sku_complete 
        ,to_location_name
        ,case when to_id = 14 then 11
            when to_id = 29 then 211
            when to_id = 31 then 221
            when to_id = 69 then 231
            when to_id = 70 then 241
            when to_id = 71 then 251
            when to_id = 72 then 261
            when to_id = 73 then 281
            when to_id = 81 then 12
            when to_id = 92 then 311
            when to_id = 93 then 341
            when to_id = 95 then 321
            when to_id = 96 then 291
            when to_id = 99 then 301
            when to_id = 102 then 361
            when to_id = 103 then 331
            when to_id = 115 then 351
            when to_id = 116 then 381
            when to_id = 130 then 15
            when to_id = 131 then 371
            when to_id = 132 then 32
            when to_id = 159 then 391
            when to_id = 163 then 35
            when to_id = 165 then 42
            when to_id = 168 then 401
            when to_id = 175 then 411
            when to_id = 177 then 431
            when to_id = 178 then 111
            when to_id = 182 then 391
            when to_id = 186 then 441
            else null
            end as id_store
        ,min(ca.year_week_id) as year_week_available
        ,min(date(date_received)) as first_available_date
        from dwh.fact_ns_transfer_order ns
        left join (select full_date
        , case when ca.week_of_year_number  in (1,2,3,4,5,6,7,8,9) 
        then concat(cast(ca.year_id as varchar),'0',CAST(ca.week_of_year_number AS VARCHAR))
             when ca.week_of_year_number = 53 or ca.week_of_year_number = 54
              then concat(cast(ca.year_id+1 as varchar),'01')
            else concat(cast(ca.year_id as varchar),cast(ca.week_of_year_number as varchar))
            end as year_week_id
        from dwh.dim_calendar ca) ca
        on  cast(ns.date_received as date) = cast(ca.full_date as date)
        left join (select id_product
        ,sku_complete 
        from dwh.dim_product dp
        where dp.date_released BETWEEN DATE_TRUNC('month', NOW() - INTERVAL '6' month) AND current_date
            and dp.id_product is not null
            and dp.original_price_th is not NULL 
            and dp.date_released is not NULL 
            and dp.parent_product_line is not null
            and dp.product_name is not null
            and dp.product_line is not null
            and dp.original_price_th != 0
            and dp.henry_category_1 not in ('Accessories', 'Bags', 'Cosmetics', 'Miscellaneous')
            and dp.parent_product_line not in ('Cosmetics','Accessories','Free Gift','3rd Party')
            and brand in ('Alita','Basics','Pomelo', 'BEET by Pomelo', 'Holiday Collection', 'Pomelo x Tex Saverio')
            and dp.id_product not in (19, 41, 14196, 14197, 14198, 14199, 14200, 14201, 14202, 14203, 14204, 14205, 14206, 14207, 14208, 17131, 17132, 17654,17182)
            and dp.active = 1) dp2
        on ns.sku_complete = dp2.sku_complete 
        where id_product is not null
        GROUP BY 1,2,3,4
         ) avl_store
        where year_week_available is not null 
        and id_store is not null
        group by avl_store.year_week_available
            ,avl_store.id_store)
        group by 1,2) pd_launched
        on cast(pd_sold.year_week_sold as varchar) = cast(pd_launched.year_week_available as varchar) 
        and cast(pd_sold.id_store as varchar) = cast(pd_launched.id_store as varchar)
        where active_product is not null
         group by 1,2,3,4,5
         order by pd_sold.year_week_sold
"""

In [4]:
sale_prop = wr.athena.read_sql_query(sql, database="dwh")

QueryFailed: SYNTAX_ERROR: line 48:12: View 'awsdatacatalog.dwh.dim_product' is stale; it must be re-created. You may need to manually clean the data at location 's3://aws-athena-query-results-207606013102-ap-southeast-1/tables/a4f76da4-f2ca-4454-a7d2-4e0e7c451b21' before retrying. Athena will not delete data in your account.

In [5]:
sale_prop["total_products"] = (
    sale_prop["tot_product_launched"] + sale_prop["tot_product_existing"]
)

In [6]:
sale_prop["tot_product_launched"] = sale_prop["tot_product_launched"].astype(float)
sale_prop["tot_product_existing"] = sale_prop["tot_product_existing"].astype(float)
sale_prop["total_products"] = sale_prop["total_products"].astype(float)

In [7]:
sale_prop["prop_new_styles"] = np.round(
    sale_prop["tot_product_launched"] / sale_prop["total_products"], 2
)
sale_prop["prop_existing_styles"] = np.round(
    sale_prop["tot_product_existing"] / sale_prop["total_products"], 2
)

In [8]:
sale_prop

,year_week_sold,year_month_id,id_shop,id_warehouse,tot_product_launched,tot_product_existing,new_pd_sold,old_pd_sold,prop_new_pd,prop_existing_pd,total_products,prop_new_styles,prop_existing_styles
0,202114,202104,2,11,0.0,0.0,0,214,0,1,0.0,NaN,NaN
1,202114,202104,5,5,55.0,0.0,167,0,1,0,55.0,1.00,0.00
2,202114,202103,2,11,12.0,0.0,212,0,1,0,12.0,1.00,0.00
3,202114,202103,2,11,0.0,0.0,0,223,0,1,0.0,NaN,NaN
4,202114,202104,2,11,12.0,0.0,241,0,1,0,12.0,1.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
359,202142,202142,1,1,33.0,630.0,18,963,0,0,663.0,0.05,0.95
360,202142,202142,11,11,40.0,510.0,24,1128,0,0,550.0,0.07,0.93
361,202142,202142,1,1,28.0,3736.0,272,9356,0,0,3764.0,0.01,0.99
362,202142,202142,1,1,35.0,6006.0,1407,32530,0,0,6041.0,0.01,0.99


In [9]:
sale_prop.rename(
    columns={
        "total_products": "tot_products",
        "prop_new_pd": "prop_new_sold",
        "prop_existing_pd": "prop_existing_sold",
        "year_week_sold": "year_week_id",
    },
    inplace=True,
)

In [10]:
sale_prop

,year_week_id,year_month_id,id_shop,id_warehouse,tot_product_launched,tot_product_existing,new_pd_sold,old_pd_sold,prop_new_sold,prop_existing_sold,tot_products,prop_new_styles,prop_existing_styles
0,202114,202104,2,11,0.0,0.0,0,214,0,1,0.0,NaN,NaN
1,202114,202104,5,5,55.0,0.0,167,0,1,0,55.0,1.00,0.00
2,202114,202103,2,11,12.0,0.0,212,0,1,0,12.0,1.00,0.00
3,202114,202103,2,11,0.0,0.0,0,223,0,1,0.0,NaN,NaN
4,202114,202104,2,11,12.0,0.0,241,0,1,0,12.0,1.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
359,202142,202142,1,1,33.0,630.0,18,963,0,0,663.0,0.05,0.95
360,202142,202142,11,11,40.0,510.0,24,1128,0,0,550.0,0.07,0.93
361,202142,202142,1,1,28.0,3736.0,272,9356,0,0,3764.0,0.01,0.99
362,202142,202142,1,1,35.0,6006.0,1407,32530,0,0,6041.0,0.01,0.99


In [12]:
sale_prop = sale_prop[
    [
        "year_week_id",
        "id_shop",
        "prop_new_sold",
        "prop_existing_sold",
        "tot_product_launched",
        "tot_product_existing",
    ]
].drop_duplicates()

In [13]:
sale_prop["id_shop"] = sale_prop["id_shop"].astype(str)

In [14]:
sale_prop  # there are duplicate becuase the data actually at id_store

,year_week_id,id_shop,prop_new_sold,prop_existing_sold,tot_product_launched,tot_product_existing
0,202114,2,1,0,12.0,0.0
2,202114,5,1,0,52.0,0.0
3,202114,5,1,0,49.0,0.0
4,202114,2,0,1,0.0,0.0
6,202115,1,1,0,14.0,196.0
...,...,...,...,...,...,...
352,202141,1,0,0,56.0,3804.0
353,202141,1,0,0,57.0,664.0
354,202141,1,0,0,34.0,1334.0
355,202141,1,0,0,43.0,854.0


In [15]:
sale_prop2 = (
    sale_prop.groupby(["year_week_id", "id_shop"]).mean().reset_index()
)  # so need to groupby first to make it at id_shop /week level

In [16]:
sale_prop2

,year_week_id,id_shop,prop_new_sold,prop_existing_sold,tot_product_launched,tot_product_existing
0,202114,2,0.5,0.5,6.000000,0.000000
1,202114,5,1.0,0.0,50.500000,0.000000
2,202115,1,0.5,0.0,15.500000,112.000000
3,202115,2,0.0,0.0,23.000000,0.000000
4,202115,5,0.0,1.0,0.000000,0.000000
...,...,...,...,...,...,...
86,202140,5,0.0,1.0,35.500000,273.000000
87,202141,1,0.0,0.0,47.181818,1085.818182
88,202141,11,0.0,0.0,24.000000,408.000000
89,202141,2,0.0,0.0,20.333333,812.666667


In [17]:
sale_prop2["new_sale_prop"] = (
    sale_prop2["prop_new_sold"] / sale_prop2["tot_product_launched"]
)
sale_prop2["new_existing_prop"] = (
    sale_prop2["prop_existing_sold"] / sale_prop2["tot_product_existing"]
)

In [18]:
sale_prop2

,year_week_id,id_shop,prop_new_sold,prop_existing_sold,tot_product_launched,tot_product_existing,new_sale_prop,new_existing_prop
0,202114,2,0.5,0.5,6.000000,0.000000,0.083333,inf
1,202114,5,1.0,0.0,50.500000,0.000000,0.019802,NaN
2,202115,1,0.5,0.0,15.500000,112.000000,0.032258,0.000000
3,202115,2,0.0,0.0,23.000000,0.000000,0.000000,NaN
4,202115,5,0.0,1.0,0.000000,0.000000,NaN,inf
...,...,...,...,...,...,...,...,...
86,202140,5,0.0,1.0,35.500000,273.000000,0.000000,0.003663
87,202141,1,0.0,0.0,47.181818,1085.818182,0.000000,0.000000
88,202141,11,0.0,0.0,24.000000,408.000000,0.000000,0.000000
89,202141,2,0.0,0.0,20.333333,812.666667,0.000000,0.000000


In [19]:
sale_prop_group = sale_prop2.groupby("id_shop")[
    [
        "prop_new_sold",
        "prop_existing_sold",
        "tot_product_launched",
        "tot_product_existing",
    ]
].mean()

In [20]:
sale_prop_group["new_sale_prop"] = (
    sale_prop_group["prop_new_sold"] / sale_prop_group["tot_product_launched"]
)
sale_prop_group["new_existing_prop"] = (
    sale_prop_group["prop_existing_sold"] / sale_prop_group["tot_product_existing"]
)

In [21]:
sale_prop_group

,prop_new_sold,prop_existing_sold,tot_product_launched,tot_product_existing,new_sale_prop,new_existing_prop
id_shop,,,,,,
1,0.038272,0.250155,20.562915,666.164818,0.001861,0.000376
11,0.000000,0.576923,10.461538,150.461538,0.000000,0.003834
2,0.017857,0.300595,19.339286,397.235119,0.000923,0.000757
5,0.043478,0.405797,20.057971,249.992754,0.002168,0.001623


In [4]:
hal = Hal()
sql = """
        select distinct main.id_store
              ,sum(main.prop_new_pd)/count(distinct main.year_week_sold) as avg_prop_new_pd
              ,sum(main.prop_existing_pd)/count(distinct main.year_week_sold) as avg_prop_existing_pd
        from (
            select distinct pd_sold.year_week_sold
                              ,pd_sold.id_store
                              ,case when sum(exist_styles.active_product) is null then 0
                                else sum(exist_styles.active_product) end as active_product
                              ,case when pd_launched.product_launched is null then 0 
                                else pd_launched.product_launched
                                end as tot_product_launched	   
                              ,case when sum(pd_sold.net_units_sold) = 0 then 0
                                else round(CAST(sum(pd_sold.new_pd_sold) AS float)/CAST(sum(pd_sold.net_units_sold) AS float),2) 
                                end as prop_new_pd
                              ,case when sum(pd_sold.net_units_sold) = 0 then 0
                                else round(CAST(sum(pd_sold.old_pd_sold) AS float)/CAST(sum(pd_sold.net_units_sold) AS float),2) 
                                end as prop_existing_pd
                        from (select distinct t.year_week_sold
                                        ,t.year_month_id
                                        ,t.id_shop
                                        ,t.id_warehouse
                                        ,t.id_store
                                        ,case when product_age = 'new' 
                                      then sum(t.tot_net_units_sold) 
                                      else 0
                                      end as new_pd_sold
                                      ,case when product_age = 'old' 
                                      then sum(t.tot_net_units_sold) 
                                      else 0
                                      end as old_pd_sold
                                      ,new_pd_sold + old_pd_sold as net_units_sold
                                from (	
                                        select dp2.id_product 
                                              ,fso2.id_shop
                                              ,case when fso2.id_store = '5b989110a9afac06f24b80ce' then '231'
                                                   when fso2.id_store = '5b55a681868e636a8ab1eaa2' then '261'
                                                   when fso2.id_store = '5bb2e6ac1e5abf2252df039c' then '251'
                                                   when fso2.id_store = '5bbedd0bc3cef17af5b1a3ab' then '11'
                                                   when fso2.id_store = '5b9890f4d49bab0743f5689e' then '241'
                                                   when fso2.id_store = '5a7c011b6878b592402d76bd' then '371'
                                                   when fso2.id_store = '39-1' then '391'
                                                   else fso2.id_store
                                                   end as id_store
                                              ,case when id_shop = '2' and year_week_id < '202017' then '1'
                                                    when id_shop = '2' and year_week_id >= '202017' then '11'
                                                    when id_shop = '11' and year_week_id < '202038' then '1'
                                                    when id_shop = '11' and year_week_id >= '202038' then '11'
                                                    WHEN id_shop = '10' THEN '1'
                                                    WHEN id_shop = '12' THEN '1'
                                                    WHEN id_shop = '4' THEN '1'
                                                     WHEN id_shop = '14' THEN '11'
                                                    else id_shop
                                                    end as id_warehouse
                                              ,ca.year_week_id as year_week_sold
                                              ,ca.year_month_id
                                              ,min(pd_avl.year_week_available) as year_week_avl
                                              ,case when year_week_avl = ca.year_week_id 
                                                    then 'new' 
                                                    else 'old'
                                                    end as product_age
                                              ,COALESCE(sum(CASE WHEN fso2.transaction_type ='Return' THEN -fso2.product_quantity ELSE fso2.product_quantity END),0) as tot_net_units_sold
                                        from dwh.fact_sales_offline fso2
                                        left join (select distinct id_product
                                                            ,sku_complete
                                                            ,date_released
                                                    from dwh.dim_product
                                                    where date_released between date('2016-12-26') and date(getdate()-1)
                                                    and id_product is not null
                                                    and original_price_th is not NULL 
                                                    and date_released is not NULL 
                                                    and parent_product_line is not null
                                                    and product_name is not null
                                                    and product_line is not null
                                                    and original_price_th != 0
                                                    and henry_category_1 not in ('Accessories', 'Bags', 'Cosmetics', 'Miscellaneous')
                                                    and parent_product_line not in ('Cosmetics','Accessories','Free Gift','3rd Party')
                                                    and brand in ('Alita','Basics','Pomelo', 'BEET by Pomelo', 'Holiday Collection', 'Pomelo x Tex Saverio')
                                                    and id_product not in (19, 41, 14196, 14197, 14198, 14199, 14200, 14201, 14202, 14203, 14204, 14205, 14206, 14207, 14208, 17131, 17132, 17654,17182)
                                                    and active = 1
                                                    ) dp2
                                        on fso2.id_product = dp2.id_product
                                        left join (select sku_complete
                                                         ,to_location_name
                                                         ,case when to_id = 14 then 11
                                                               when to_id = 29 then 211
                                                               when to_id = 31 then 221
                                                               when to_id = 69 then 231
                                                               when to_id = 70 then 241
                                                               when to_id = 71 then 251
                                                               when to_id = 72 then 261
                                                               when to_id = 73 then 281
                                                               when to_id = 81 then 12
                                                               when to_id = 92 then 311
                                                               when to_id = 93 then 341
                                                               when to_id = 95 then 321
                                                               when to_id = 96 then 291
                                                               when to_id = 99 then 301
                                                               when to_id = 102 then 361
                                                               when to_id = 103 then 331
                                                               when to_id = 115 then 351
                                                               when to_id = 116 then 381
                                                               when to_id = 130 then 15
                                                               when to_id = 131 then 371
                                                               when to_id = 132 then 32
                                                               when to_id = 159 then 391
                                                               when to_id = 163 then 35
                                                               when to_id = 165 then 42
                                                               when to_id = 168 then 401
                                                               when to_id = 175 then 411
                                                               when to_id = 177 then 431
                                                               when to_id = 178 then 111
                                                               when to_id = 182 then 391
                                                               when to_id = 186 then 441
                                                          else null
                                                          end as id_store
                                                         ,min(ca.year_week_id) as year_week_available
                                                         ,min(date(date_received)) as first_available_date
                                                    FROM dwh.fact_ns_transfer_order ns
                                                    left join (select full_date
                                                                            , case when ca.week_of_year_number  in (1,2,3,4,5,6,7,8,9) 
                                                                            then concat(convert(varchar(4),ca.year_id),RIGHT('0'+CAST(ca.week_of_year_number AS VARCHAR(2)),2))
                                                                            when ca.week_of_year_number = 53
                                                                            then concat(convert(varchar(4),(ca.year_id+1)),'01')
                                                                            else concat(ca.year_id,ca.week_of_year_number) end as year_week_id
                                                                     from dwh.dim_calendar ca) ca
                                                    on  date(ns.date_received) = ca.full_date 
                                                    where id_store is not null
                                                    GROUP BY 1,2,3
                                                    order by sku_complete
                                                    ) pd_avl
                                        on dp2.sku_complete = pd_avl.sku_complete and pd_avl.id_store = fso2.id_store 
                                        left join (select full_date
                                                            ,case when ca.month_id  in (1,2,3,4,5,6,7,8,9) 
                                           then concat(convert(varchar(4),ca.year_id),RIGHT('0'+CAST(ca.month_id AS VARCHAR(2)),2))
                                        else concat(ca.year_id,ca.week_of_year_number) end as year_month_id
                                                                , case when ca.week_of_year_number  in (1,2,3,4,5,6,7,8,9) 
                                                                then concat(convert(varchar(4),ca.year_id),RIGHT('0'+CAST(ca.week_of_year_number AS VARCHAR(2)),2))
                                                                when ca.week_of_year_number = 53
                                                                then concat(convert(varchar(4),(ca.year_id+1)),'01')
                                                                else concat(ca.year_id,ca.week_of_year_number) end as year_week_id
                                                            from dwh.dim_calendar ca) ca
                                        on  date(fso2.transaction_time) = ca.full_date
                                        where fso2.revenue_usd >0
                                        and transaction_time between date(getdate()-180) and date(getdate()-1)
                                        and sales_id is not null
                                        and dp2.id_product is not null
                                        and pd_avl.year_week_available is not null
                                        group by dp2.id_product
                                                ,fso2.id_shop
                                                ,fso2.id_store
                                                ,ca.year_week_id
                                                ,ca.year_month_id
                                        having year_week_sold >= year_week_avl
                                        ) t
                                group by t.year_week_sold
                                      ,t.id_shop
                                      ,t.year_month_id
                                      ,t.id_warehouse
                                      ,t.id_store
                                      ,product_age
                                    )pd_sold
                        left join (select ca.year_week_id
                                        ,case when fisom.id_shop is null then '1'
                                        else fisom.id_shop 
                                        end as id_shop
                                        ,case when id_store = '5b989110a9afac06f24b80ce' then '231'
                                           when id_store = '5b55a681868e636a8ab1eaa2' then '261'
                                           when id_store = '5bb2e6ac1e5abf2252df039c' then '251'
                                           when id_store = '5bbedd0bc3cef17af5b1a3ab' then '11'
                                           when id_store = '5b9890f4d49bab0743f5689e' then '241'
                                           when id_store = '5a7c011b6878b592402d76bd' then '371'
                                           else id_store
                                           end as id_store
                                        ,count(distinct dp2.id_product) as active_product
                                from dwh.fact_inventory_snapshot_offline_master fisom 
                                left join (select distinct id_product
                                                        from dwh.dim_product dp
                                                        where dp.date_released between date('2016-12-26') and date(getdate()-1)
                                                        and dp.id_product is not null
                                                        and dp.original_price_th is not NULL 
                                                        and dp.date_released is not NULL 
                                                        and dp.parent_product_line is not null
                                                        and dp.product_name is not null
                                                        and dp.product_line is not null
                                                        and dp.original_price_th != 0
                                                        and dp.henry_category_1 not in ('Accessories', 'Bags', 'Cosmetics', 'Miscellaneous')
                                                        and dp.parent_product_line not in ('Cosmetics','Accessories','Free Gift','3rd Party')
                                                        and brand in ('Alita','Basics','Pomelo', 'BEET by Pomelo', 'Holiday Collection', 'Pomelo x Tex Saverio')
                                                        and dp.id_product not in (19, 41, 14196, 14197, 14198, 14199, 14200, 14201, 14202, 14203, 14204, 14205, 14206, 14207, 14208, 17131, 17132, 17654,17182)
                                                        and dp.active = 1) dp2
                                on fisom.id_product = dp2.id_product	
                                left join (select full_date
                                                        , case when ca.week_of_year_number  in (1,2,3,4,5,6,7,8,9) 
                                                          then concat(convert(varchar(4),ca.year_id),RIGHT('0'+CAST(ca.week_of_year_number AS VARCHAR(2)),2))
                                                          when ca.week_of_year_number = 53
                                                          then concat(convert(varchar(4),(ca.year_id+1)),'01')
                                                          else concat(ca.year_id,ca.week_of_year_number) 
                                                          end as year_week_id
                                                    from dwh.dim_calendar ca) ca
                                on fisom.snapshot_date = ca.full_date
                                where active = 1
                                and fisom.snapshot_date between date('2018-06-22') and date(getdate()-1)
                                and id_store not in ('1','201', '25', '271')	
                                group by id_shop
                                        ,id_store
                                        , ca.year_week_id
                        ) exist_styles
                        on pd_sold.year_week_sold = exist_styles.year_week_id and pd_sold.id_store = exist_styles.id_store
                        left join (select distinct year_week_available
                                          ,id_store
                                          ,sum(tot_product_launched) as product_launched 
                                    from (select distinct avl_store.year_week_available
                                              ,avl_store.id_store
                                              ,count(distinct avl_store.id_product) as tot_product_launched
                                        from (select dp2.id_product
                                                   ,dp2.sku_complete 
                                                   ,to_location_name
                                                   ,case when to_id = 14 then 11
                                                       when to_id = 29 then 211
                                                       when to_id = 31 then 221
                                                       when to_id = 69 then 231
                                                       when to_id = 70 then 241
                                                       when to_id = 71 then 251
                                                       when to_id = 72 then 261
                                                       when to_id = 73 then 281
                                                       when to_id = 81 then 12
                                                       when to_id = 92 then 311
                                                       when to_id = 93 then 341
                                                       when to_id = 95 then 321
                                                       when to_id = 96 then 291
                                                       when to_id = 99 then 301
                                                       when to_id = 102 then 361
                                                       when to_id = 103 then 331
                                                       when to_id = 115 then 351
                                                       when to_id = 116 then 381
                                                       when to_id = 130 then 15
                                                       when to_id = 131 then 371
                                                       when to_id = 132 then 32
                                                       when to_id = 159 then 391
                                                       when to_id = 163 then 35
                                                       when to_id = 165 then 42
                                                       when to_id = 168 then 401
                                                       when to_id = 175 then 411
                                                       when to_id = 177 then 431
                                                       when to_id = 178 then 111
                                                       when to_id = 182 then 391
                                                       when to_id = 186 then 441
                                                              else null
                                                              end as id_store
                                                    ,min(ca.year_week_id) as year_week_available
                                                    ,min(date(date_received)) as first_available_date
                                                FROM dwh.fact_ns_transfer_order ns
                                                left join (select full_date
                                                                , case when ca.week_of_year_number  in (1,2,3,4,5,6,7,8,9) 
                                                                then concat(convert(varchar(4),ca.year_id),RIGHT('0'+CAST(ca.week_of_year_number AS VARCHAR(2)),2))
                                                                when ca.week_of_year_number = 53
                                                                then concat(convert(varchar(4),(ca.year_id+1)),'01')
                                                                else concat(ca.year_id,ca.week_of_year_number) end as year_week_id
                                                            from dwh.dim_calendar ca) ca
                                                on  date(ns.date_received) = ca.full_date 
                                                left join (select id_product
                                                                 ,sku_complete 
                                                           from dwh.dim_product dp
                                                           where dp.date_released between date('2016-12-26') and date(getdate()-1)
                                                                and dp.id_product is not null
                                                                and dp.original_price_th is not NULL 
                                                                and dp.date_released is not NULL 
                                                                and dp.parent_product_line is not null
                                                                and dp.product_name is not null
                                                                and dp.product_line is not null
                                                                and dp.original_price_th != 0
                                                                and dp.henry_category_1 not in ('Accessories', 'Bags', 'Cosmetics', 'Miscellaneous')
                                                                -- ,'Lingerie Bodysuit', 'Sportswear Bottoms', 'Sportswear Outerwear', 'Sportswear Tops', 'Swimwear')
                                                                and dp.parent_product_line not in ('Cosmetics','Accessories','Free Gift','3rd Party')
                                                                and brand in ('Alita','Basics','Pomelo', 'BEET by Pomelo', 'Holiday Collection', 'Pomelo x Tex Saverio')
                                                                and dp.id_product not in (19, 41, 14196, 14197, 14198, 14199, 14200, 14201, 14202, 14203, 14204, 14205, 14206, 14207, 14208, 17131, 17132, 17654,17182)
                                                                and dp.active = 1
                                                          ) dp2
                                                on ns.sku_complete = dp2.sku_complete 
                                                where id_store is not null
                                                and id_product is not null
                                                GROUP BY 1,2,3,4
                                                ) avl_store
                                        where year_week_available is not null 
                                        and id_store is not null
                                        group by avl_store.year_week_available
                                              ,avl_store.id_store)
                                    group by 1,2
                                ) pd_launched
                        on pd_sold.year_week_sold = pd_launched.year_week_available and pd_sold.id_store = pd_launched.id_store
                        where active_product is not null
                        group by pd_sold.year_week_sold
                                ,pd_sold.id_store
                                ,pd_launched.product_launched
        ) main
        group by main.id_store
        order by main.id_store
        """
store_dist_6mths = hal.get_pandas_df(sql)

Insert your Github token to access Vault:


GithubToken ········································


In [5]:
store_dist_6mths

,id_store,avg_prop_new_pd,avg_prop_existing_pd
0,11,0.100500,0.899500
1,111,0.032000,0.968000
2,12,0.054444,0.945556
3,15,0.048636,0.951364
4,211,0.072000,0.928000
5,221,0.167500,0.832500
6,231,0.000000,1.000000
7,251,0.090000,0.910000
8,261,0.130909,0.869091
9,281,0.196500,0.803500


In [6]:
# if import error run this cell first
!pip install --upgrade google-api-python-client oauth2client
!pip install gspread

In [7]:
import boto3

s3 = boto3.resource("s3")

In [8]:
#### authenticate your identity using the JSON credentials and provide access to your Google Drive #####
from apiclient import discovery
from httplib2 import Http
import oauth2client
from oauth2client import file, client, tools

obj = lambda: None
lmao = {
    "auth_host_name": "localhost",
    "noauth_local_webserver": "store_true",
    "auth_host_port": [8080, 8090],
    "logging_level": "ERROR",
}
for k, v in lmao.items():
    setattr(obj, k, v)

# authorization boilerplate code
SCOPES = "https://www.googleapis.com/auth/drive.readonly"
store = file.Storage("token.json")
creds = store.get()
# creds = None
# The following will give you a link if token.json does not exist, the link allows the user to give this app permission
if not creds or creds.invalid:
    flow = client.flow_from_clientsecrets(
        "/home/ec2-user/SageMaker/business-intelligence-notebooks/dfm_clothing/online_dfm_v2/client_secret.json",
        SCOPES,
    )
    creds = tools.run_flow(flow, store, obj)

In [9]:
import io
from googleapiclient.http import MediaIoBaseDownload

DRIVE = discovery.build("drive", "v3", http=creds.authorize(Http()))
# https://docs.google.com/spreadsheets/d/1LKibyPJvNLSvPR4A5t3zcdEBPAxrFuy4/edit#gid=1796573573
file_id = "1LKibyPJvNLSvPR4A5t3zcdEBPAxrFuy4"
request = DRIVE.files().get_media(fileId=file_id)
# replace the filename and extension in the first field below
fh = io.FileIO("store_traffic.xlsx", mode="w")
downloader = MediaIoBaseDownload(fh, request)
done = False
while done is False:
    status, done = downloader.next_chunk()
    print("Download %d%%." % int(status.progress() * 100))

Download 100%.


In [10]:
df = pd.read_excel(
    "store_traffic.xlsx", engine="openpyxl", sheet_name="unpivot_forecast"
)

In [19]:
store_map = df[["Store_name", "id_store", "oldcluster", "country"]].drop_duplicates()

In [23]:
store_map = store_map[~store_map["id_store"].isna()]

In [25]:
store_map["id_store"] = store_map["id_store"].astype(int)
store_map["id_store"] = store_map["id_store"].astype(str)

/home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/ipykernel/__main__.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  if __name__ == '__main__':
/home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/ipykernel/__main__.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  from ipykernel import kernelapp as app


In [26]:
store_map

,Store_name,id_store,oldcluster,country
0,Central World,261,sc2,TH
7,Icon Siam,11,sc2,TH
14,EmQuartier,251,sc2,TH
21,Mega Bangna,211,sc1,TH
28,Central Pinklao,221,sc3,TH
35,SG 313 Somerset,12,sc1,SG
42,Central Phuket,301,sc3,TH
49,Central Rama 9,311,sc3,TH
56,Central Rama 3,321,sc3,TH
63,Zpell,331,sc3,TH


In [27]:
store_dist_6mths

,id_store,avg_prop_new_pd,avg_prop_existing_pd
0,11,0.100500,0.899500
1,111,0.032000,0.968000
2,12,0.054444,0.945556
3,15,0.048636,0.951364
4,211,0.072000,0.928000
5,221,0.167500,0.832500
6,231,0.000000,1.000000
7,251,0.090000,0.910000
8,261,0.130909,0.869091
9,281,0.196500,0.803500


In [30]:
store_dist_6mths = store_dist_6mths.merge(
    store_map[["id_store", "oldcluster", "country"]], how="left", on="id_store"
)

In [34]:
store_dist_6mths = store_dist_6mths[~store_dist_6mths["country"].isna()]

In [35]:
store_dist_6mths

,id_store,avg_prop_new_pd,avg_prop_existing_pd,oldcluster,country
0,11,0.100500,0.899500,sc2,TH
1,111,0.032000,0.968000,sc2,MY
2,12,0.054444,0.945556,sc1,SG
3,15,0.048636,0.951364,sc3,ID
4,211,0.072000,0.928000,sc1,TH
5,221,0.167500,0.832500,sc3,TH
7,251,0.090000,0.910000,sc2,TH
8,261,0.130909,0.869091,sc2,TH
11,301,0.033704,0.966296,sc3,TH
12,311,0.116000,0.884000,sc3,TH


In [36]:
store_dist_6mths.to_csv(
    "s3://hal-bi-bucket/data_science/dfm/offline_clothing_v2/data/deploy_store_dist_offline.csv",
    index=False,
)